# 📦 Notebook 1 – Environment Setup

**Purpose:** Install all packages, mount Google Drive, check GPU, set global paths, and extract the dataset.

**Output:** A `config.json` file saved to `/content/` that all other notebooks read to know where the data lives.

**Run this notebook first, before any other notebook.**

---
Pipeline order:
```
01_setup  →  02_preprocess  →  03_yolo_train  →  04_pose_train  →  05_evaluate  →  06_visualize
```

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive |
| 2 | Install all required packages |
| 3 | Verify GPU and print hardware info |
| 4 | Define and save global configuration |
| 5 | Clone the project repository from GitHub |
| 6 | Extract the LineMOD dataset zip |

## Cell 1 – Mount Google Drive

We mount Drive so we can:
- Read the dataset zip (`Linemod_preprocessed.zip`) stored on Drive
- Save trained model checkpoints back to Drive (survives session resets)

After running, click the authorization link that appears and paste the code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

## Cell 2 – Install all required packages (single call)

Installing in one call is faster than multiple `!pip install` lines because pip resolves all dependencies at once instead of repeatedly.

| Package | Purpose |
|---------|----------|
| `torch / torchvision / torchaudio` | PyTorch with CUDA 11.8 support |
| `ultralytics` | YOLOv8 object detection |
| `open3d` | 3D point cloud, mesh loading, ICP |
| `opencv-python`, `Pillow` | Image I/O |
| `scikit-image` | Additional image processing |
| `scipy` | Quaternion / rotation math |
| `numpy`, `pandas`, `matplotlib` | Numerics, tables, plots |
| `tqdm` | Progress bars |
| `pyyaml` | Read `gt.yml` config files |
| `colorama` | Coloured terminal output in training logs |
| `numba` | JIT-compile radius-map kernel (10–100× faster) |

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q ultralytics open3d opencv-python Pillow scikit-image scipy \
              numpy pandas matplotlib tqdm pyyaml colorama numba

print('✅ All packages installed.')

## Cell 3 – Verify GPU and print hardware info

If `torch.cuda.is_available()` returns `False`:
- Go to **Runtime → Change runtime type → Hardware accelerator → GPU**

Best Colab GPUs for this project (from fastest to slowest):
`A100 (80 GB) > V100 (16 GB) > A100 (40 GB) > T4 (16 GB)`

In [ ]:
import torch

print('─' * 55)
print(f'  PyTorch version : {torch.__version__}')
print(f'  CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU model       : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  VRAM            : {vram:.1f} GB')
    print(f'  CUDA version    : {torch.version.cuda}')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print('  ⚠️  No GPU detected! Change runtime to GPU before continuing.')
print('─' * 55)

## Cell 4 – Define and save global configuration

All path constants and shared settings are defined here and saved to `/content/config.json`. Every other notebook loads this file at startup so they always use consistent paths — no copy-pasting paths between notebooks.

To change any path (e.g. if your zip is named differently), edit **only this cell**.

`ADD_THRESHOLDS`: per-class success thresholds for the ADD metric (metres). Source: standard LineMOD benchmark values.

In [ ]:
import os, json, numpy as np

DRIVE_LINEMOD_ROOT = '/content/drive/MyDrive/Machine Learning Project/linemod'

CONFIG = {
    'DATASET_ZIP'    : '/content/drive/MyDrive/Linemod_preprocessed.zip',
    'LINEMOD_ROOT'   : DRIVE_LINEMOD_ROOT,
    'DATA_DIR'       : f'{DRIVE_LINEMOD_ROOT}/Linemod_preprocessed/data',
    'MODELS_DIR_PLY' : f'{DRIVE_LINEMOD_ROOT}/Linemod_preprocessed/models',
    'YOLO_DIR'       : '/content/dataset/linemod/Linemod_ready',
    'DRIVE_MODELS'   : '/content/drive/MyDrive/models',
    'REPO_DIR'       : '/content/6D-pose-estimation',
    'ALL_CLASSES'    : ['01','02','04','05','06','08','09','10','11','12','13','14','15'],
    'CLASS_NAMES'    : ['ape','benchvise','can','cat','driller',
                        'duck','eggbox','glue','holepuncher','iron','lamp','phone','camera'],
    'CAMERA_K'       : [[572.4114, 0.0, 325.2611],
                        [0.0, 573.57043, 242.04899],
                        [0.0, 0.0, 1.0]],
    'ADD_THRESHOLDS' : {
        '01':0.01421,'02':0.03309,'04':0.02223,'05':0.02842,
        '06':0.01859,'08':0.03188,'09':0.01557,'10':0.01974,
        '11':0.01931,'12':0.01961,'13':0.03172,'14':0.03166,'15':0.02543
    },
}

# Dataset already lives extracted on Drive (DATA_DIR above), so we only need
# to make sure the local output folder for trained models exists.
os.makedirs(CONFIG['DRIVE_MODELS'], exist_ok=True)

CONFIG_PATH = '/content/config.json'
with open(CONFIG_PATH, 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'✅ Config saved to {CONFIG_PATH}')
print(f'   DATA_DIR    → {CONFIG["DATA_DIR"]}')
print(f'   DRIVE_MODELS → {CONFIG["DRIVE_MODELS"]}')

## Cell 5 – Clone the project repository from GitHub

The repo contains `src/model.py` and `src/dataset.py` which are shared modules imported by the training and evaluation notebooks.

If already cloned (e.g. from a previous run), the cell skips the clone step and does a `git pull` instead.

In [ ]:
REPO_DIR = CONFIG['REPO_DIR']

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Sina-Ghiabi/6D-Pose-Estimation.git {REPO_DIR}
    print(f'✅ Repository cloned to {REPO_DIR}')
else:
    !cd {REPO_DIR} && git pull
    print(f'✅ Repository updated at {REPO_DIR}')

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('✅ sys.path updated — src modules are importable.')

## Cell 6 – Extract the LineMOD dataset zip

Unzips `Linemod_preprocessed.zip` from Drive to `/content/dataset/linemod/`.

**Expected zip structure:**
```
Linemod_preprocessed/data/01/rgb/*.png
Linemod_preprocessed/data/01/depth/*.png
Linemod_preprocessed/data/01/mask/*.png
Linemod_preprocessed/data/01/gt.yml
Linemod_preprocessed/models/obj_01.ply ... obj_15.ply
```

If the data was already extracted in a previous session, this cell is skipped automatically (checks for existence of `DATA_DIR`).

In [ ]:
import zipfile
from tqdm import tqdm

DATA_DIR = CONFIG['DATA_DIR']

if os.path.isdir(DATA_DIR):
    print(f'⏩ Dataset already at {DATA_DIR} — skipping extraction.')
else:
    zip_path   = CONFIG['DATASET_ZIP']
    extract_to = CONFIG['LINEMOD_ROOT']
    print(f'📦 Extracting {zip_path} → {extract_to}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.infolist()
        for m in tqdm(members, desc='Extracting', unit='file'):
            zf.extract(m, extract_to)
    print('\n✅ Extraction complete.')

found = sorted(d for d in os.listdir(DATA_DIR)
               if os.path.isdir(os.path.join(DATA_DIR, d)))
print(f'\nClass folders found: {found}')

---
## ✅ Setup Complete

- ✅ Google Drive mounted
- ✅ All packages installed
- ✅ GPU verified
- ✅ Config saved to `/content/config.json`
- ✅ Repository cloned
- ✅ Dataset extracted

**Next step → Open and run `02_preprocess.ipynb`**